## Successful Semantic Modelling for Power BI - "Lab 02. Storage modes"

Module 2 in the browser. Replaces what would otherwise be a DAX Studio demo, so
nobody needs a desktop tool, the right version, or admin rights to follow along.

WHAT IT SHOWS - run the cells IN ORDER, each one builds on the last

ACT 1  Direct Lake paging (cells 4-9)
cold -> query -> more resident -> query -> more resident -> reframe -> cold
plus the trap that ClearCache does NOT undo any of it.
ACT 2  DirectQuery (cell 10) - what a DirectQueryEnd event looks like.
ACT 3  Hybrid (cells 11-14) - the same query hits or skips the DirectQuery
partition depending purely on how you write the filter.

Run as a **Python** notebook. It is all DAX, DMV and REST, no Spark session.

In [ ]:
# ---- CELL 1: INSTALL --------------------------------------------------------
# Alone, and first. In Fabric %pip restarts the Python interpreter, so anything
# defined before it is lost. Config and imports therefore live in Cell 2.
%pip install -q semantic-link-labs

In [ ]:
# ---- CELL 2: CONFIG + IMPORTS ----------------------------------------------
WORKSPACE = None                              # None = this notebook's workspace
DL_MODEL  = "01 Star Schema (fixed)"          # Direct Lake, for Act 1
DQ_MODEL  = "02 Storage - DirectQuery"        # 100% DirectQuery, for Act 2
HY_MODEL  = "04 Scaling - Hybrid"             # import + DirectQuery. Demo now lives in Module 4

import sempy.fabric as fabric
import sempy_labs as labs
import pandas as pd
import time
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

WORKSPACE = WORKSPACE or fabric.resolve_workspace_name()
print(f"workspace: {WORKSPACE}")
for m in (DL_MODEL, DQ_MODEL, HY_MODEL):
    print(f"  {m}")

In [ ]:
# ---- CELL 3: HELPER - which columns are in memory right now? ---------------
# Direct Lake pages column segments in on demand. This DMV is the ground truth:
# ISRESIDENT says the segment is in memory, TEMPERATURE how recently it was used.
SEGMENTS_DMV = """
SELECT [TABLE_ID], [COLUMN_ID], [ISRESIDENT], [TEMPERATURE], [USED_SIZE]
FROM $SYSTEM.DISCOVER_STORAGE_TABLE_COLUMN_SEGMENTS
"""

def resident(model, label=""):
    """Print the columns currently paged into memory. Returns the DataFrame."""
    try:
        df = fabric.evaluate_dax(dataset=model, workspace=WORKSPACE, dax_string=SEGMENTS_DMV)
    except Exception as e:
        print(f"  DMV unavailable ({str(e)[:120]}) - fall back to the Delta Analyzer demo")
        return pd.DataFrame()

    df.columns = [c.split("[")[-1].rstrip("]") for c in df.columns]
    df = df[~df["COLUMN_ID"].astype(str).str.startswith("RowNumber")]
    hot = df[df["ISRESIDENT"].astype(str).str.lower().isin(["true", "1"])]
    print(f"{label}: {len(hot)} of {len(df)} column segments resident")
    if len(hot):
        print(hot[["TABLE_ID", "COLUMN_ID", "TEMPERATURE"]]
              .sort_values("TEMPERATURE", ascending=False).head(15).to_string(index=False))
    return hot


# Server Timings in a notebook. QueryEnd = the whole query, VertiPaqSEQueryEnd =
# storage engine scans, DirectQueryEnd = a round trip to the source. That last
# one is what Acts 2 and 3 are about: we want to SEE it, then make it disappear.
EVENTS = {
    "QueryEnd":           ["EventClass", "EventSubclass", "TextData", "Duration", "CpuTime"],
    "VertiPaqSEQueryEnd": ["EventClass", "EventSubclass", "TextData", "Duration", "CpuTime"],
    "DirectQueryEnd":     ["EventClass", "TextData", "Duration", "CpuTime"],
}

def _col(df, name):
    key = name.replace(" ", "").lower()
    for c in df.columns:
        if c.replace(" ", "").lower() == key:
            return c
    return None

def run(model, dax, label="query", settle=5, show_sql=False):
    """Run a DAX query with a trace and report SE scans vs DirectQuery events."""
    with fabric.create_trace_connection(dataset=model, workspace=WORKSPACE) as tc:
        with tc.create_trace(EVENTS, "Storage modes") as tr:
            tr.start()
            result = fabric.evaluate_dax(dataset=model, workspace=WORKSPACE, dax_string=dax)
            time.sleep(settle)
            logs = tr.stop()

    ec, sub, dur = _col(logs, "EventClass"), _col(logs, "EventSubclass"), _col(logs, "Duration")
    if ec is None:
        display(logs)
        return result

    q  = logs[logs[ec] == "QueryEnd"]
    se = logs[logs[ec] == "VertiPaqSEQueryEnd"]
    dq = logs[logs[ec] == "DirectQueryEnd"]
    if sub is not None:                       # drop the internal duplicates
        se = se[~se[sub].astype(str).str.contains("Internal", case=False, na=False)]

    total  = float(q[dur].max() or 0)
    dq_ms  = float(dq[dur].sum() or 0)
    verdict = "DirectQuery partition WAS queried" if len(dq) else "no DirectQuery, served from memory"
    print(f"{label}")
    print(f"  total {total:>6.0f} ms | SE scans {len(se):>2} | DirectQuery events {len(dq):>2} ({dq_ms:.0f} ms)  -> {verdict}")
    if show_sql and len(dq):
        print(dq[_col(logs, "TextData")].iloc[0][:600])
    return result

In [ ]:
# ---- CELL 4: ACT 1 step 1 - start from genuinely cold ----------------------
# Getting back to cold means REFRAMING, not clearing the cache. A reframe points
# the model at the latest Delta version and invalidates the in-memory cache, so
# every column has to be paged in again. Seconds, and no data is copied.
# Expect: 0 resident.
labs.refresh_semantic_model(dataset=DL_MODEL, workspace=WORKSPACE)
cold = resident(DL_MODEL, "1. COLD (after reframe)")

In [ ]:
# ---- CELL 5: ACT 1 step 2 - one measure, one column ------------------------
# The measure only needs sales[SalesAmount]. Nothing else should appear.
run(DL_MODEL, 'EVALUATE ROW("Total", [Total Sales])', "one measure")
warm1 = resident(DL_MODEL, "2. AFTER one measure")

In [ ]:
# ---- CELL 6: ACT 1 step 3 - add grouping columns ---------------------------
# Now we need two dimension columns and a second measure as well. PREDICT how
# many segments will be resident before you run it. That is the exercise.
run(DL_MODEL, """
EVALUATE
SUMMARIZECOLUMNS (
    'Date'[MonthYear],
    'Product'[Category],
    "Sales", [Total Sales],
    "Qty",   [Total Quantity]
)
""", "grouped query")
warm2 = resident(DL_MODEL, "3. AFTER a grouped query")

print()
print(f"resident segments: {len(cold)} -> {len(warm1)} -> {len(warm2)}")
print("Only the columns each query touched were paged in. That is column-on-demand.")

In [ ]:
# ---- CELL 7: ACT 1 step 4 - the whole model at a glance --------------------
# VertiPaq Analyzer as an interactive HTML report, right here in the notebook.
#
# read_stats_from_data=True fetches column cardinality from the data. On Direct
# Lake that means querying the Delta tables. Costs a few seconds, hence opt-in.
#
# READ THIS AS A MEMORY PICTURE. On Direct Lake it describes what VertiPaq holds
# *right now*, which is why cells 4-6 came first. For the on-DISK story (Parquet
# rowgroups, file count, V-Order) use labs.delta_analyzer, as Module 1 does.
vpa = labs.vertipaq_analyzer(dataset=DL_MODEL, workspace=WORKSPACE,
                             read_stats_from_data=True)   # dark_mode=True projects better

print("sections returned:", list(vpa))   # each one is a DataFrame you can sort and filter

In [ ]:
# ---- CELL 8: ACT 1 step 5 - ClearCache does NOT undo any of this -----------
# Two different caches, and only one of them evicts columns:
#
#   ClearCache  drops the storage engine's cached query RESULTS. Useful before
#               timing a query so you measure real work. Columns stay resident.
#   Reframe     invalidates the in-memory column cache itself.
#
# Expect the count to be UNCHANGED from cell 6.
labs.clear_cache(dataset=DL_MODEL, workspace=WORKSPACE)
after_clear = resident(DL_MODEL, "4. AFTER clear_cache")
print(f"  still {len(after_clear)} resident. Query results were dropped, columns were not.")

In [ ]:
# ---- CELL 9: ACT 1 step 6 - reframe DOES, closing the loop -----------------
# Expect 0 again, back where cell 4 started.
labs.refresh_semantic_model(dataset=DL_MODEL, workspace=WORKSPACE)
after_reframe = resident(DL_MODEL, "5. AFTER reframe")
print(f"  now {len(after_reframe)} resident. Reframing is what evicts columns.")

In [ ]:
# ---- CELL 10: ACT 2 - what DirectQuery actually looks like -----------------
# Same shape of query, but a 100% DirectQuery model. Every request becomes SQL,
# so expect DirectQuery events > 0 and no VertiPaq scans worth speaking of.
# Nothing is cached in memory, which is the trade: always current, never instant.
run(DQ_MODEL, """
EVALUATE
SUMMARIZECOLUMNS ( 'date'[MonthYear], "Sales", [Total Sales] )
""", "DirectQuery model", show_sql=True)

In [ ]:
# ---- CELL 11: ACT 3a - Hybrid, asking only for the hot slice ---------------
# 04 Scaling - Hybrid holds OrderDateKey >= 20260101 in an IMPORT partition and
# everything older in a DIRECTQUERY partition. The DirectQuery partition carries
# a data coverage definition, so the engine can skip it when a query cannot
# possibly need it. Expect: 0 DirectQuery events.
HYBRID_BASE = """
EVALUATE
SUMMARIZECOLUMNS (
    'date'[MonthYear],
    'product'[Brand],
    %s,
    "Total Sales", [Total Sales]
)
"""

run(HY_MODEL, HYBRID_BASE % "FILTER ( VALUES ( 'sales'[OrderDateKey] ), 'sales'[OrderDateKey] >= 20260101 )",
    "hot slice only  (>= 20260101)")

In [ ]:
# ---- CELL 12: ACT 3b - Hybrid, asking for history --------------------------
# Now the query genuinely needs the cold partition, so the engine has to go and
# get it. Expect: DirectQuery events > 0. This is correct behaviour, not a bug.
run(HY_MODEL, HYBRID_BASE % "FILTER ( VALUES ( 'sales'[OrderDateKey] ), 'sales'[OrderDateKey] < 20260101 )",
    "cold history   (<  20260101)")

In [ ]:
# ---- CELL 13: ACT 3c - the trap that quietly costs you the whole benefit ----
# Logically identical to cell 11, and it returns the right answer. But DATE()
# returns a DATETIME, which the engine holds as a DOUBLE (46023.0 for 2026-01-01),
# so comparing it to an int64 key promotes the whole comparison to floating point.
# The predicate becomes ">= 46023", true for every yyyymmdd key, so it filters
# nothing AND the coverage definition can no longer be matched.
# Expect: DirectQuery events > 0, exactly what cell 11 avoided.
run(HY_MODEL, HYBRID_BASE % "FILTER ( VALUES ( 'sales'[OrderDateKey] ), 'sales'[OrderDateKey] >= DATE ( 2026, 1, 1 ) )",
    "the DATE() trap")

In [ ]:
# ---- CELL 14: ACT 3d - filtering a different column entirely ---------------
# No filter on the fact at all. The coverage definition is RELATED('date'[DateKey])
# < 20260101, and 'date' is a DUAL table, so the engine resolves Year = 2026
# against its in-memory copy, turns that into a range of DateKeys, and sees the
# cold partition cannot contribute. Expect: 0 DirectQuery events.
# This is the one to dwell on - the filter and the coverage definition do not
# share a column, and it still works.
run(HY_MODEL, HYBRID_BASE % "TREATAS ( { 2026 }, 'date'[Year] )",
    "dual-dim filter (Year = 2026)")

In [ ]:
# ---- CELL 15: REFERENCE - sources, fallback and guardrails -----------------
# Where the Direct Lake model reads from, whether anything would fall back, and
# the SKU limits that decide when Direct Lake stops fitting in memory.
for src in labs.directlake.get_direct_lake_sources(dataset=DL_MODEL, workspace=WORKSPACE):
    route = "SQL endpoint" if src.get("usesSqlEndpoint") else "OneLake (direct)"
    print(f"  {src.get('itemName')} ({src.get('itemType')}) -> {route}")

display(labs.directlake.check_fallback_reason(dataset=DL_MODEL, workspace=WORKSPACE))

sku = labs.directlake.get_sku_size(workspace=WORKSPACE)
print(f"Capacity SKU: {sku}")
display(labs.directlake.get_directlake_guardrails_for_sku(sku))